# Build and Verify GCN

In [ ]:
#importing foundation
import torch
import copy
import numpy as np
import torch.nn as nn
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
#creating fake data
n = 100
y = torch.randint(0,2,(n,))
means = torch.tensor([[0,0,0],[1,1,1]])
noise = torch.randn(n,3) * 1.5
x = means[y] + noise
permutations = torch.randperm(n)
train_mask = torch.zeros(n,dtype=torch.bool)
train_mask[permutations[:60]]= True
val_mask = torch.zeros(n,dtype=torch.bool)
val_mask[permutations[60:80]] = True
test_mask = torch.zeros(n,dtype=torch.bool)
test_mask[permutations[80:]] = True

In [ ]:
#creating graph
p_in, p_out = 0.10, 0.01
same = (y[:,None]) == y[None:,]
prob = torch.where(same,p_in,p_out)
A = (torch.rand(n,n) < prob).float()
A = torch.triu(A,diagonal=1)
A = A + A.T

**Normalizing adjecency using the formula**   
Normalized adjecency = Diagonal inverse square root  * A tilda * Diagonal inverse square root  

Equals to: A(ij) = Node Ai connected with Node Aj / square root of its diagonal


In [ ]:
def normalized_adj(A):
  A_tilde = A + torch.eye(A.shape[0])
  deg = A_tilde.sum(dim=1)
  d_inv_sqt = deg.pow(-0.5)
  return d_inv_sqt[:,None] * A_tilde * d_inv_sqt[None:,]

GCN layer and the model

In [ ]:
class GCNLayer(nn.Module):
  def __init__(self,in_dim,out_dim):
    super().__init__()
    self.W = nn.Parameter(torch.empty(in_dim,out_dim))
    self.b = nn.Parameter(torch.zeros(out_dim))
    nn.init.xavier_uniform_(self.W)



  def forward(self,A_hat,H):
    return A_hat @ (H @ self.W) + self.b

class GCN(nn.Module):
  def __init__(self,in_dim,hidden,out_dim):
    super().__init__()
    self.l1 = GCNLayer(in_dim,hidden)
    self.l2 = GCNLayer(hidden,out_dim)
  def forward(self,A,X):
    A_hat = normalized_adj(A)
    h = torch.relu(self.l1(A_hat,X))
    return self.l2(A_hat,h)


The MLP layer

In [ ]:
class MLP(nn.Module):
  def __init__(self,in_dim,hidden,out_dim):
    super().__init__()
    self.f1 = nn.Linear(in_dim,hidden)
    self.f2 = nn.Linear(hidden,out_dim)
  def forward(self,x):
    x = torch.relu(self.f1(x))
    return self.f2(x)

Training the model

In [ ]:
def train_model(model,forward,epochs=300, lr = 0.01, wd = 5e-4):
  loss_fn = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),lr=lr, weight_decay=wd)
  def accuracy(logits,y,mask):
    return (logits[mask].argmax(dim=1) ==y[mask]).float().mean().item()
  best_val_loss, best_state,best_epoch = float('inf'), None, 0
  for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = forward(model)
    loss = loss_fn(logits[train_mask],y[train_mask])
    loss.backward()
    optimizer.step()

    with torch.no_grad():
      logits = forward(model)
      val_accuracy = accuracy(logits,y,val_mask)
      val_loss = loss_fn(logits[val_mask],y[val_mask]).item()
    if val_loss < best_val_loss:
        best_val_loss,best_state,best_epoch, = val_loss, copy.deepcopy(model.state_dict()), epoch
    if(epoch % 20 == 0):
      print(f"Epoch {epoch:3d} loss {loss.item():.3f} val_accuracy {val_accuracy:.2f}")
  model.load_state_dict(best_state)
  model.eval()
  print("best epoch:", best_epoch)
  with torch.no_grad():
      logits = forward(model)
  return {"best_epoch": best_epoch, "train": accuracy(logits, y, train_mask),
            "val": accuracy(logits, y, val_mask), "test": accuracy(logits, y, test_mask)}
gcn = GCN(3,16,2)
gcn_result = train_model(gcn,lambda m: m(A,x))
print("GCN",gcn_result)
print("MLM", train_model(MLP(3,16,2),lambda m: m(x)))

Epoch   0 loss 0.887 val_accuracy 0.45
Epoch  20 loss 0.432 val_accuracy 0.65
Epoch  40 loss 0.245 val_accuracy 0.95
Epoch  60 loss 0.164 val_accuracy 1.00
Epoch  80 loss 0.141 val_accuracy 1.00
Epoch 100 loss 0.134 val_accuracy 1.00
Epoch 120 loss 0.131 val_accuracy 1.00
Epoch 140 loss 0.129 val_accuracy 1.00
Epoch 160 loss 0.128 val_accuracy 1.00
Epoch 180 loss 0.127 val_accuracy 1.00
Epoch 200 loss 0.126 val_accuracy 1.00
Epoch 220 loss 0.126 val_accuracy 1.00
Epoch 240 loss 0.125 val_accuracy 1.00
Epoch 260 loss 0.124 val_accuracy 1.00
Epoch 280 loss 0.123 val_accuracy 1.00
best epoch: 284
GCN {'best_epoch': 284, 'train': 0.9666666388511658, 'val': 1.0, 'test': 1.0}
Epoch   0 loss 0.656 val_accuracy 0.45
Epoch  20 loss 0.474 val_accuracy 0.60
Epoch  40 loss 0.449 val_accuracy 0.60
Epoch  60 loss 0.426 val_accuracy 0.60
Epoch  80 loss 0.399 val_accuracy 0.45
Epoch 100 loss 0.372 val_accuracy 0.50
Epoch 120 loss 0.347 val_accuracy 0.50
Epoch 140 loss 0.326 val_accuracy 0.50
Epoch 160

Manual Verification

In [ ]:
model = GCN(3,16,2)
model.eval()
A_hat = normalized_adj(A)
with torch.no_grad():
  layer = model.l1
  actual = layer(A_hat,x)[0]

  manual_output = torch.zeros(layer.W.shape[1])
  for j in range(n):
    transformed_features = x[j] @ layer.W
    manual_output += A_hat[0,j] * transformed_features
  manual_output += layer.b
print(f"Matrix output {actual}")
print(f"Manual output {manual_output}")

print("Match:",torch.allclose(actual,manual_output,atol=1e-6))

Matrix output tensor([-0.5534,  0.1578, -0.3813,  0.5309, -0.4490, -0.1060,  0.1629, -0.3985,
         0.2076,  0.3667,  0.5261,  0.9090, -0.6678,  0.0683, -0.2184,  0.1166])
Manual output tensor([-0.5534,  0.1578, -0.3813,  0.5309, -0.4490, -0.1060,  0.1629, -0.3985,
         0.2076,  0.3667,  0.5261,  0.9090, -0.6678,  0.0683, -0.2184,  0.1166])
Match: True


Verifying permutations equivarence by changing the row of the graph then the columns consquently to match the connections

In [ ]:
gcn.eval()
with torch.no_grad():
  original_logits = gcn(A,x)
perm = torch.randperm(n)
A_perm = A[perm][:,perm]
x_perm = x[perm]
with torch.no_grad():
  perm_logits =  gcn(A_perm, x_perm)
expected_logits = original_logits[perm]

# checking weather they are equal
print(torch.allclose(perm_logits,expected_logits,atol=1e-5))

True


checking conenction also removed since gcn is highly dependent of eatch node connection with other. Used 0.8 or 80% to see noticable difference. So now related to MLP since GNN without graph connection is MLP


In [ ]:
A_removed = A.clone()
edges = torch.triu(A,diagonal=1).nonzero()
num_remove = int(0.8 * len(edges))
selected = torch.randperm(len(edges))[:num_remove]
edges_to_remove = edges[selected]
for i, j in edges_to_remove:
  A_removed[i,j] = 0
  A_removed[j,i] = 0
gcn.eval()
with torch.no_grad():
  original_logits = gcn(A,x)
  removed_logits = gcn(A_removed,x)
original_acc = (original_logits[test_mask].argmax(dim=1) == y[test_mask]).float().mean().item()
removed_acc = (removed_logits[test_mask].argmax(dim=1) == y[test_mask]).float().mean().item()

print("Original test accuracy:",original_acc)
print("After edge removal:",removed_acc)

Original test accuracy: 1.0
After edge removal: 0.800000011920929


Random graph connection by making the ability of gnn to lose since it is random connection not related by the probability making it close to MLP since they are random

In [ ]:
A_random = torch.zeros_like(A)
all_possible_edges = torch.triu(torch.ones(n,n),diagonal=1).nonzero()
num_edges = len(edges)
chosen = torch.randperm(len(all_possible_edges))[:num_edges]

random_edges = all_possible_edges[chosen]
for i,j in random_edges:
  A_random[i,j] = 1
  A_random[j,i] = 1
with torch.no_grad():
  original_logits = gcn(A,x)
  random_logits = gcn(A_random,x)
original_acc = (original_logits[test_mask].argmax(dim=1) == y[test_mask]).float().mean().item()
random_acc = (random_logits[test_mask].argmax(dim=1) == y[test_mask]).float().mean().item()

print("Original test accuracy:",original_acc)
print("After randomized connection:",random_acc)

Original test accuracy: 1.0
After randomized connection: 0.8999999761581421
